In [13]:
from ingest import load_faq_data

documents = load_faq_data()
print(len(documents))

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

1350


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
from tqdm.auto import tqdm
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

#len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [24]:
import numpy as np

X = np.array(vectors)
#X.shape
query="Can I still join the course after the start date?"
v_query=model.encode(query)
scores=X.dot(v_query)
idx = np.argmax(scores) 
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [27]:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 538, 907, 625,   2])

In [30]:
top5[::-1]
scores[top5]

#top5 = np.argsort(-scores)[:5]

array([0.56009996, 0.6536313 , 0.71921325, 0.7579372 , 0.7629411 ],
      dtype=float32)

In [40]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [42]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}